# 07 — Evaluation (Corrected)
No silent checkpoint fallback. Missing trained variants are skipped and explicitly reported. Results are smoke-test results until the full evaluation set is run.

In [ ]:
!pip install -q peft transformers datasets pandas
import os,sys,shutil,torch,pandas as pd
repo=os.path.abspath(os.getcwd())
if os.path.isdir('/kaggle/input') and not os.path.exists(os.path.join(repo,'src')):
    for root,dirs,files in os.walk('/kaggle/input'):
        if 'src' in dirs and os.path.exists(os.path.join(root,'src','evaluation','metrics.py')):
            shutil.copytree(os.path.join(root,'src'),os.path.join('/kaggle/working','src'),dirs_exist_ok=True); repo='/kaggle/working'; break
sys.path.insert(0,repo)
from datasets import load_dataset
from transformers import AutoModelForCausalLM,AutoTokenizer
from peft import PeftModel
from src.evaluation.metrics import extract_eval_test_cases
from src.debugging.debug_loop import agentic_debug_loop
BASE='deepseek-ai/deepseek-coder-1.3b-instruct'
he=load_dataset('openai_humaneval',split='test'); mbpp=load_dataset('mbpp',split='test')
print('Benchmarks:',len(he),'HumanEval /',len(mbpp),'MBPP')

In [ ]:
def adapter_path(tag):
    p=f'./checkpoints/{tag}/final'
    required=['adapter_config.json','adapter_model.safetensors']
    if tag=='dpo': required.append('dpo_metadata.json')
    if tag=='ppo': required.append('ppo_metadata.json')
    return p if all(os.path.isfile(os.path.join(p,f)) for f in required) else None
def load_variant(tag):
    tok=AutoTokenizer.from_pretrained(BASE,trust_remote_code=True)
    model=AutoModelForCausalLM.from_pretrained(BASE,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,device_map='auto' if torch.cuda.is_available() else None,trust_remote_code=True)
    if tag!='zero_shot':
        p=adapter_path(tag)
        if p is None: return None,tok
        model=PeftModel.from_pretrained(model,p,is_trainable=False)
    model.eval(); return model,tok
def evaluate_max_k(model, tokenizer, dataset, max_k=5, label=''):
    total=len(dataset); pass1=0; solved_at={1:0,3:0,5:0}
    for i,example in enumerate(dataset):
        tests=extract_eval_test_cases(example)
        if not tests: raise ValueError(f'No executable tests for evaluation example {i}')
        problem=example.get('prompt',example.get('question',example.get('text','')))
        history=agentic_debug_loop(model,tokenizer,problem,tests,K=max_k)
        statuses=[h['result']['status'] for h in history]
        pass1 += int(bool(statuses) and statuses[0]=='AC')
        for k in (1,3,5):
            solved_at[k] += int('AC' in statuses[:min(k,len(statuses))])
        if (i+1)%5==0 or i+1==total:
            print(f'{label} [{i+1}/{total}] Pass@1={pass1/(i+1):.2%} Fix@3={solved_at[3]/(i+1):.2%} Fix@5={solved_at[5]/(i+1):.2%}')
    return {'total':total,'pass_at_1':pass1/total if total else 0.0,'fix_at_1':solved_at[1]/total if total else 0.0,'fix_at_3':solved_at[3]/total if total else 0.0,'fix_at_5':solved_at[5]/total if total else 0.0}
rows=[]
for tag,label in [('zero_shot','Zero-Shot'),('sft','SFT'),('ppo','PPO'),('dpo','DPO')]:
    model,tok=load_variant(tag)
    if model is None: print('SKIPPED:',label,'checkpoint missing or incomplete'); continue
    h=evaluate_max_k(model,tok,he.select(range(min(20,len(he)))),max_k=5,label=label+' HE')
    m=evaluate_max_k(model,tok,mbpp.select(range(min(20,len(mbpp)))),max_k=5,label=label+' MBPP')
    rows.append({'model':label,'K':1,'N_HE':h['total'],'N_MBPP':m['total'],'HE_Pass@1':h['pass_at_1'],'HE_Fix@K':h['fix_at_1'],'MBPP_Pass@1':m['pass_at_1'],'MBPP_Fix@K':m['fix_at_1']})
    for k in (3,5):
        rows.append({'model':label,'K':k,'N_HE':h['total'],'N_MBPP':m['total'],'HE_Pass@1':h['pass_at_1'],'HE_Fix@K':h[f'fix_at_{k}'],'MBPP_Pass@1':m['pass_at_1'],'MBPP_Fix@K':m[f'fix_at_{k}']})
    del model; import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
df=pd.DataFrame(rows); print(df.to_string(index=False)); df.to_csv('evaluation_results_corrected.csv',index=False)
assert all(df.groupby('model')['HE_Fix@K'].apply(lambda s: list(s)==sorted(s))) if len(df) else True
assert all(df.groupby('model')['MBPP_Fix@K'].apply(lambda s: list(s)==sorted(s))) if len(df) else True
print('These are smoke-test results (N<=20), not final paper results.')

### Required final runs
Use the same fixed benchmark subset, seed, decoding settings, and executable tests for every available checkpoint. Run the full evaluation only after PPO/DPO checkpoints have been independently verified. Do not reuse earlier simulated values.